# GPT-Neo inference with the HF's Transformers Library
This notebook is a companion of chapter 4 of the "Domain Specific LLMs in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is to introduce readers to the inference (text generation) with the [GPT-Neo model](https://github.com/EleutherAI/gpt-neo) using the Hugging Face's [Transformers library](https://github.com/huggingface/transformers). It can be executed in the Colab free tier with hardware acceleration (GPU).  
More details about the code can be found in the book's chapter.

Install the missing requirements in the Colab VM (HF's Accelerate only).

In [1]:
!pip install accelerate

Download the GPT-Neo 2.7B model and the associated tokenizer from the HF's Hub. The model is loaded in full precision and is then loaded into the GPU.

In [2]:
import torch
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "EleutherAI/gpt-neo-2.7B"
tokenizer = GPT2Tokenizer.from_pretrained(model_id)
model = GPTNeoForCausalLM.from_pretrained(model_id, device_map="auto")
#model.to(device)

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.7GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/420 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-2.7B
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...30}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0...31}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Verify where the model layers have been loaded (all in the GPU memory or also RAM and/or disk).

In [3]:
model.hf_device_map

AttributeError: 'GPTNeoForCausalLM' object has no attribute 'hf_device_map'

Perform standard inference (text completion).

In [4]:
prompt = "The story so far: in the beginning, the universe was created."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(input_ids,
                               do_sample=True,
                               temperature=0.9,
                               max_length=200,
                               pad_token_id=50256)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)

The story so far: in the beginning, the universe was created. There was no time, just space. Then, the big bang. Then, the whole universe exploded. There was nothing to be found. A place to be found.

Then, there was nothing to find.

In our universe, there’s a hole. A black hole. A “black hole” as it’s called in the physics community. It’s a hole in space, and it’s a hole that sucks matter into itself, making it dense. That’s what makes it a black hole.

Scientists are trying to solve the mystery of that black hole, because the thing keeps sucking up matter and it’s sucking up matter at an accelerating rate.

That makes space-time expand, which is why the universe keeps getting bigger and bigger — the vacuum gets pulled apart just like a balloon when you pop the stopper off


Do few-shot text classification (the model can generalize learning from few new and unseen examples.

In [5]:
prompt = """
Sentence: This movie is very nice.
Sentiment: positive

#####

Sentence: I hated this movie, it sucks.
Sentiment: negative

#####

Sentence: This movie was actually pretty funny.
Sentiment: positive

#####

Sentence: This movie could have been better.
Sentiment: neutral
"""
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(input_ids,
                               do_sample=True,
                               temperature=0.9,
                               max_length=200,
                               pad_token_id=50256)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)


Sentence: This movie is very nice.
Sentiment: positive

#####

Sentence: I hated this movie, it sucks.
Sentiment: negative

#####

Sentence: This movie was actually pretty funny.
Sentiment: positive

#####

Sentence: This movie could have been better.
Sentiment: neutral

#####

Sentence: This movie was very clever and witty.
Sentiment: positive

#####

Sentence: I really enjoyed this movie.
Sentiment: neutral

#####

Sentence: I liked this movie very much.
Sentiment: neutral

#####

Sentence: This movie was very interesting.
Sentiment: positive

#####

Sentence: This movie is very exciting.
Sentiment: neutral

#####

Sentence: This movie was a very engaging movie.



Do Python code generation.

In [6]:
prompt = """Instruction: Generate a Python function that lets you reverse a list of integers.

Answer: """
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(input_ids,
                               do_sample=True,
                               temperature=0.9,
                               max_length=200,
                               pad_token_id=50256
                               )
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)

Instruction: Generate a Python function that lets you reverse a list of integers.

Answer: 
Use the 'for x in' loop to go through each number, then the same pattern, but with a 'list = []' at the beginning. The output looks like the input:
For numbers between 0 and 5 (inclusive):
    For number in [1, 2, 3, 4, 5]:
        number > 6

Now, for numbers between 6 and 10 (inclusive) with a value of 11:
    For number in [11]:
        number < 6

With this for loop, you can see what it does:
For numbers between 0 and 5 (inclusive):
    For number in [1, 2, 3, 4, 5]:
        number > 6
number > 6


Do batch text completion.

In [7]:
texts = ["Once there was a man ", "The weather today will be ", "A great soccer player must "]

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
encoding = tokenizer(texts, padding=True, return_tensors='pt').to(device)
with torch.no_grad():
    generated_ids = model.generate(**encoding,
                                   do_sample=True,
                                   temperature=0.9,
                                   max_length=50,
                                   pad_token_id=50256)
generated_texts = tokenizer.batch_decode(
    generated_ids, skip_special_tokens=True)

for text in generated_texts:
  print("---------")
  print(text)

---------
Once there was a man  
Who never wanted to be a man  
He never made it until he died.

And if I had not wanted to be a man  
When I was first born  
I
---------
The weather today will be 
beautiful, in the 60's, and 
it looks to be a clear day 
and you'll have to 
be in the air in most 
cases. In addition, we'll 
---------
A great soccer player must  
Be able to keep his balance as he runs.  
But one man who has done so is  
Futbolista, the man with one foot.

#   



Benchmarking the model on text completion: comparing the cases where the KV cache is used to those where it isn't.

In [8]:
import time
import numpy as np

prompt = "The story so far: in the beginning, the universe was created."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

for use_cache in (True, False):
  times = []
  for _ in range(20):
    start = time.time()
    generated_ids = model.generate(input_ids,
                                  do_sample=True,
                                  temperature=0.9,
                                  max_length=200,
                                  pad_token_id=50256,
                                  use_cache=use_cache)
    times.append(time.time() - start)
  print(f"{'Using' if use_cache else 'No'} KV cache: {round(np.mean(times), 3)} +- {round(np.std(times), 3)} seconds")

Using KV cache: 10.952 +- 2.053 seconds
No KV cache: 52.534 +- 0.176 seconds


Benchmarking the model's total generation time.

In [ ]:
import time
import numpy as np

prompt = "The story so far: in the beginning, the universe was created."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

max_length = 300
times = []
inference_runs = 21
for _ in range(inference_runs):
  start = time.time()
  generated_ids = model.generate(input_ids,
                                do_sample=True,
                                temperature=0.9,
                                max_length=max_length,
                                pad_token_id=50256,
                                )
  times.append(time.time() - start)
print(f"Average Total Generation time: {round(np.mean(times[1:]), 3)} +- {round(np.std(times[1:]), 3)} seconds")